# Retries and Reflection

**Prerequisites:** `02_the_agent_loop.ipynb`, and from `01_foundations`: `05_structured_outputs.ipynb`, `06_function_calling.ipynb`

Notebook 2's loop has exactly one way of dealing with a bad turn: run out of budget and stop. That's a safety net, not a strategy — it prevents disaster without ever recovering from anything.

There are two genuinely different things that go wrong, and they need different answers:

- **The call was malformed.** The right tool, the wrong arguments. Nothing about the plan is wrong; the request just needs fixing. This is mechanical, and the loop shouldn't even see it.
- **The call succeeded and the answer still isn't good.** Nothing raised. There is no error to catch. Only judgment can tell.

This notebook builds one for each: `DebugThink`, which repairs arguments inside `Act` where the loop never notices, and `ReflectionThink`, which critiques a finished answer and hands the verdict back to the loop.

**By the end you'll have:**
- Watched a tool call fail permanently, then watched the identical call succeed once `Act` was allowed to repair it — and seen why the repair needs its own `Think` rather than re-asking the one that made the mistake
- Understood why `DebugThink` is deliberately not a loop component, and what would break if it were
- Built `ReflectionThink` as critique-only — it never writes the answer, and its output isn't even a decision
- Built `Loop`, a bounded inner cycle, and used `count` plus `repeat_from` to run a few rounds, critique them, and re-enter the same loop with the critique in hand

## Setup

Same two tools as notebook 2, and the same imports plus the two Thinks this notebook is about.

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
sys.path.append(str(Path.cwd().parent.parent / "01_foundations"))

from chat import Conversation
from runtime import Runtime, Loop
from observe import Observe
from think import Think, ReflectionThink, DebugThink
from act import Act

In [2]:
from datetime import datetime
from typing import Literal
from zoneinfo import ZoneInfo, ZoneInfoNotFoundError

from pydantic import BaseModel, field_validator

def get_current_time(timezone):
    return datetime.now(ZoneInfo(timezone)).strftime("%Y-%m-%d %H:%M %Z")

def convert_temperature(value, from_unit, to_unit):
    if from_unit.upper() == "F" and to_unit.upper() == "C":
        return round((value - 32) * 5 / 9, 1)
    if from_unit.upper() == "C" and to_unit.upper() == "F":
        return round(value * 9 / 5 + 32, 1)
    raise ValueError(f"unsupported conversion: {from_unit} to {to_unit}")

class GetCurrentTimeArgs(BaseModel):
    timezone: str

    @field_validator("timezone")
    @classmethod
    def timezone_must_be_valid(cls, v):
        try:
            ZoneInfo(v)
        except ZoneInfoNotFoundError:
            raise ValueError(f"'{v}' is not a valid IANA timezone")
        return v

class ConvertTemperatureArgs(BaseModel):
    value: float
    from_unit: Literal["C", "F"]
    to_unit: Literal["C", "F"]

FUNCTIONS = {"get_current_time": get_current_time, "convert_temperature": convert_temperature}
SCHEMAS = {"get_current_time": GetCurrentTimeArgs, "convert_temperature": ConvertTemperatureArgs}
tools = [
    {"type": "function", "function": {
        "name": "get_current_time",
        "description": "Get the current date and time in a given IANA timezone.",
        "parameters": {
            "type": "object",
            "properties": {"timezone": {"type": "string", "description": "IANA timezone name, e.g. Asia/Tokyo"}},
            "required": ["timezone"],
        },
    }},
    {"type": "function", "function": {
        "name": "convert_temperature",
        "description": "Convert a temperature value between Celsius and Fahrenheit.",
        "parameters": {
            "type": "object",
            "properties": {
                "value": {"type": "number"},
                "from_unit": {"type": "string", "enum": ["C", "F"]},
                "to_unit": {"type": "string", "enum": ["C", "F"]},
            },
            "required": ["value", "from_unit", "to_unit"],
        },
    }},
]

## A call that fails for a fixable reason

`"Tokyo"` is not an IANA timezone. `"Asia/Tokyo"` is. It's the kind of mistake a model makes constantly, and `GetCurrentTimeArgs`'s custom validator (chapter 1, notebook 6) catches it before `get_current_time` ever runs.

With `retries=1`, `Act` gets one attempt and no more:

In [3]:
one_shot = Act(functions=FUNCTIONS, schemas=SCHEMAS, retries=1)
print(one_shot.execute("get_current_time", '{"timezone": "Tokyo"}'))

{'tool': 'get_current_time', 'content': "Error: permanently failed after 1 attempts: 1 validation error for GetCurrentTimeArgs\ntimezone\n  Value error, 'Tokyo' is not a valid IANA timezone [type=value_error, input_value='Tokyo', input_type=str]\n    For further information visit https://errors.pydantic.dev/2.13/v/value_error"}


A permanent failure, returned rather than raised — `Act` never throws at the loop, because an error the loop can read is more useful than an exception that unwinds it. That's chapter 1's principle from notebook 9, unchanged: a failure that lands in the conversation can be recovered from; one that propagates as an exception can't.

Now the same call, with `Act` allowed three attempts:

In [4]:
retrying = Act(functions=FUNCTIONS, schemas=SCHEMAS, retries=3)
print(retrying.execute("get_current_time", '{"timezone": "Tokyo"}'))

{'tool': 'get_current_time', 'content': '2026-08-10 20:50 JST'}


Same tool, same starting arguments, real answer. Something between attempt 1 and attempt 2 turned `"Tokyo"` into `"Asia/Tokyo"`.

## `DebugThink`: repair as its own job

```python
def execute(self, tool_name, args):
    debugger = DebugThink(self.schemas)
    attempts = []

    for attempt in range(self.retries):
        tool_call = SimpleNamespace(id="call", function=SimpleNamespace(name=tool_name, arguments=args))
        try:
            result = execute_tool_call(tool_call, self.functions, self.schemas)
            return {"tool": tool_name, "content": result}
        except Exception as e:
            attempts.append({"args": args, "error": str(e)})
            if attempt == self.retries - 1:
                return {"tool": tool_name,
                        "content": f"Error: permanently failed after {self.retries} attempts: {e}"}
            args = debugger.fix(tool_name, args, str(e), attempts)
```

The loop body is ordinary. What matters is `attempts`, which accumulates every `(args, error)` pair and is passed to every subsequent repair:

```python
class DebugThink:
    def __init__(self, schemas):
        self.schemas = schemas

    def fix(self, tool_name, args, error, attempts):
        schema = self.schemas[tool_name]
        history = "\n".join(f"- args {a['args']} failed with: {a['error']}" for a in attempts)
        corrected = extract(
            f"Calling the tool `{tool_name}` with arguments {args} failed with this error:\n{error}\n\n"
            f"Earlier failed attempts on this same call:\n{history}\n\n"
            f"Fix the arguments to match this JSON schema:\n{schema.model_json_schema()}\n"
            "Return corrected JSON matching the schema.",
            schema,
        )
        return corrected.model_dump_json()
```

Three deliberate choices here, and each is a change from how chapter 1 handled the same failure.

**It sees the schema, not the conversation.** Chapter 1's `resolve_tools()` recovered by appending the error to the messages and calling the same model again with the same tools and the same history. That works, but it's asking a component that just made a mistake to notice its own mistake, buried in a page of context. `DebugThink` gets `model_json_schema()` and nothing else — the exact contract the arguments have to satisfy. The task shrinks from "keep going with this conversation" to "make this JSON valid," which is a much smaller thing to get right.

**It sees every prior attempt, not just the last one.** Without `attempts`, three retries can produce the same wrong guess three times. With it, the prompt literally contains the list of things already known not to work.

**It returns via `extract()`, so the repair is validated before it's used.** `extract()` (chapter 1, notebook 5) parses into `schema` and retries on its own if the model returns something unparseable — so a repair that is itself malformed never reaches `execute_tool_call()`.

### Why `DebugThink` is not a loop component

Every other Think in this chapter has `run(conversation)` and gets declared into a `Runtime`. `DebugThink` has `fix(...)` and is constructed privately inside `Act.execute()`. That asymmetry is on purpose.

A malformed argument is not a fact about the task. Nothing downstream benefits from knowing that `"Tokyo"` was tried before `"Asia/Tokyo"` — it's noise that would sit in the context window forever, on every future turn. Worse, notebook 2's `_doom_loop` watches for the same tool call repeating; a repair loop writing near-identical decisions into `messages` would trip it, and the runtime would shut down a recovery that was working.

So `Act.execute()` returns exactly one thing: a result, or a permanent failure. What it took to get there stays inside.

This is also why `execute()` is a **pure** method — it reads and writes nothing shared, only its own locals. That doesn't matter yet. It matters in notebook 4, where several tool calls run at the same time in different threads.

## The other failure: nothing raised, and the answer is still wrong

`DebugThink` handles the case where something threw. The harder case is when nothing does.

Here is a conversation where a tool ran, returned cleanly, and the result is simply wrong — 17 × 24 is 408, not 300. There is no exception to catch, no schema violated, no error string anywhere. Every mechanical check in this chapter passes:

In [5]:
wrong = Conversation()
wrong.messages.append({"role": "user", "content": "What is 17 * 24?"})
wrong.messages.append({"role": "decision", "type": "tool_call", "tool": "multiply", "content": '{"a": 17, "b": 24}'})
wrong.messages.append({"role": "tool", "content": "The result is 300."})

critique = ReflectionThink().run(wrong)
for key, value in critique.items():
    print(f"{key:12} {value!r}")

role         'critique'
content      'The result provided correctly calculates the product of 17 and 24 as 408, but the tool stated it as 300, which is incorrect. The decision to multiply the numbers was appropriate but the execution was flawed.'
improvements 'To improve, the calculation should be redone with attention to accuracy. The correct calculation is 17 * 24 = 408. Additionally, implementing a method to double-check calculations or using a reliable calculator could prevent such mistakes in the future.'
good_enough  False


`good_enough: False` — on a result that raised nothing, violated no schema, and produced no error string. Every mechanical check in this chapter passes on that conversation; only judgment reaches a verdict.

This run caught the real problem: 17 x 24 is 408, the tool said 300, and the critique says so and shows the correct arithmetic. That is exactly what you want.

It is worth knowing it does not always. An earlier run of this same cell returned `good_enough: False` for quite different reasons — that the tool didn't restate which operation it performed, and that `a` and `b` are uninformative argument names. Both fair, neither the glaring error sitting in front of it. Reflection reliably notices that *something* is wrong here; which thing it fastens on varies run to run.

That variance is the reason for the design in the next section. A critique is evidence for a later `Think` to weigh, not a verdict to act on — and a `Think` that disagrees with a weak critique can carry on regardless, which it could not do if the critic were also writing the answer.

```python
class Reflection(BaseModel):
    critique: str
    improvements: str
    good_enough: bool

class ReflectionThink(Think):
    def run(self, conversation):
        result = extract(
            f"Conversation so far:\n{render(conversation)}\n\n"
            "Critique whether the most recent result actually answers the original request and "
            "was carried out well. Say what was good and what wasn't. If it's not good enough, "
            "describe concrete improvements to try next. Do not write the final answer yourself.\n"
            f"Return JSON matching this schema:\n{Reflection.model_json_schema()}",
            Reflection,
        )
        return {
            "role": "critique",
            "content": result.critique,
            "improvements": result.improvements,
            "good_enough": result.good_enough,
        }
```

### It refuses to write the answer, and that's the whole design

The earlier version of this class had a fourth field, `final_answer`, and returned a `respond` decision. It critiqued the work *and* fixed it in one step. That seems more efficient and is actually worse, for two reasons.

**A critic that rewrites has no incentive to be honest.** Asked to judge and to produce the deliverable in the same breath, the easiest path is a mild critique and a lightly polished version of what's already there. Splitting the jobs means the critique is the only output, so it has to stand on its own.

**Rewriting isn't reflection's job — it's the loop's.** If the critic can produce a final answer, then the critic decides when the work is done, and it can only ever apply fixes it can make itself in one turn. A critique that says "the temperature conversion is missing" needs another *tool call* to resolve, which a rewriting critic cannot do.

So `ReflectionThink` returns `{"role": "critique", ...}` — not a `decision`. There is nothing for `Act` to execute. The critique lands in the conversation, and the next `Think` reads it as ordinary context and makes its own ordinary decision: call another tool, or answer. Given an unreliable critique, that `Think` is free to disagree with it — which it could not do if the critique arrived as a finished answer.

The prompt ends by embedding `Reflection.model_json_schema()` rather than referring vaguely to "the given schema". Field names the model has to invent are field names it will invent *differently*, and `extract()`'s retry loop shouldn't be doing work a literal schema in the prompt makes unnecessary. `DebugThink` above does the same.

That's why there's no retry machinery anywhere in this notebook for the reflection case. "Try again" isn't a mechanism; it's what a normal `Think` naturally does when the conversation it's reading contains a critique saying the last attempt fell short. `critique` joins `decision` and `plan` as a role with no API equivalent — notebook 2's `_build_messages()` narrates it as the agent's own prior turn, which is exactly what it is.

## `Loop`: a bounded cycle inside the loop

Reflection wants a shape `Runtime` can't express on its own: *do some work, then critique it.* `repeat_from` cycles a suffix of the declared list forever — it has no notion of "these three components, a few times, then move on."

```python
class Loop:
    def __init__(self, components, until=None, count=None, max_iterations=50):
        assert until is not None or count is not None, "Loop needs either `until` or `count`"
        self.components = list(components)
        self.until = until
        self.count = count
        self.max_iterations = max_iterations

    def run(self, conversation):
        passes = 0
        for _ in range(self.max_iterations):
            for component in self.components:
                output = component.run(conversation)
                if output is not None:
                    conversation.messages.append(output)
                if conversation.messages and conversation.messages[-1].get("role") == "assistant":
                    return None
            passes += 1
            if self.count is not None and passes >= self.count:
                return None
            if self.until is not None and self.until(conversation):
                return None
        return None
```

`Loop` has the same `run(conversation)` interface as everything else, so `Runtime` can't tell it apart from an `Observe` — it's a component that happens to contain components.

**Two ways to stop, for two different situations.** `count` is a plain number of passes, for when you just want a few rounds. `until` is a predicate over the conversation, for when the stopping condition depends on what actually happened — notebook 4 uses it. Either works alone; both can be given together.

**Both are checked after a complete pass, never mid-pass.** This was a real bug: checking first meant the loop could see a decision that a `Think` had just produced but that `Act` hadn't executed yet, and exit — abandoning the final step one beat before it ran.

**The one thing that does stop it mid-pass is an `assistant` message.** If a `Think` decides to respond on the first of three planned rounds, there is nothing left to do, and no predicate would necessarily have noticed — "the work is finished" and "the loop's stopping condition is met" are different claims.

**It gets none of `Runtime`'s protection.** No budgets, no doom-loop detection. `max_iterations=50` is a hard ceiling so a bug here can't hang forever, but real safety stays with the outer `Runtime`, which is still counting every component either of them runs. `Loop` also returns `None` and says nothing when it finishes — unlike a budget stop, handing control back is normal, not an event worth reporting.

## Putting them together

Two rounds of ReAct, then a critique, then back into the same loop:

In [6]:
runtime = Runtime(
    loop=[
        Observe(event="What's the current time in Tokyo, and convert 100 degrees Fahrenheit to Celsius?"),
        Loop(
            [Think(tools=tools, allow_fork=False), Act(functions=FUNCTIONS, schemas=SCHEMAS), Observe()],
            count=2,
        ),
        ReflectionThink(),
    ],
    repeat_from=1,
)

conversation = runtime.run(Conversation())
for m in conversation.messages:
    extra = {k: v for k, v in m.items() if k not in ("role", "content")}
    print(f"- {m['role']:9} {m['content']!r}" + (f"\n            {extra}" if extra else ""))

- user      "What's the current time in Tokyo, and convert 100 degrees Fahrenheit to Celsius?"
- decision  '{"timezone":"Asia/Tokyo"}'
            {'type': 'tool_call', 'tool': 'get_current_time', 'reasoning': 'To get the current time in Tokyo, I need to call the function to get the current time in a specific timezone.'}
- tool      'The current time in Tokyo is 20:50.'
- decision  '{"from_unit":"F","to_unit":"C","value":100}'
            {'type': 'tool_call', 'tool': 'convert_temperature', 'reasoning': 'I will call the function to convert the temperature from Fahrenheit to Celsius.'}
- tool      '100 degrees Fahrenheit is equivalent to 37.8 degrees Celsius.'
- critique  "The most recent result does answer the original request, but it was carried out in two separate steps. What was good is that the tool was able to provide the current time in Tokyo and convert 100 degrees Fahrenheit to Celsius. However, what wasn't good is that the original request had two distinct questions and the to

Trace what `repeat_from=1` actually did. Position 0 (the request) ran once. Position 1 is the `Loop`, which ran two full `Think → Act → Observe` passes and handed back. Position 2 critiqued them. Then the runtime walked off the end of the list and wrapped to index 1 — back into the *same* `Loop` object.

On that second entry, its `Think` saw the critique sitting in the conversation and decided to respond. `Act` turned that into an `assistant` message, `Loop`'s mid-pass check returned immediately, and `Runtime` saw the role and stopped. The second pass of that re-entered loop never ran, and never needed to.

Nothing here is retry machinery. There is no "if not good_enough, go again" branch anywhere — not in `Runtime`, not in `Loop`, not in `ReflectionThink`. The critique is just context, and a `Think` reading context and deciding is the thing `Think` already does.

The alternative design — a `Think`/`Act` pair declared *after* `ReflectionThink` to handle the retry — would have worked for exactly one round of it. `repeat_from` pointing back at the loop supports any number, and reuses components that were already there.

## Self-Refine: critiquing prose instead of work

The loop above wraps `Think -> Act -> Observe` in the `Loop`, so its critique judges *work* — several turns of tool calls and results. That is Reflexion.

Self-Refine judges *prose*. The model drafts, reads its own draft, and rewrites. No tool runs, so there is nothing for `Act` to execute and nothing for `Observe` to report, and `ReflectionThink` goes straight after `Think`:

```
Loop[ Think -> ReflectionThink ]        prose      (Self-Refine)
Loop[ Think -> Act -> Observe ] -> ReflectionThink  work   (Reflexion)
```

What makes the drafting loop legal is that `Think` produces a **decision**, and a decision is only an intention. Nothing ends a run until an `Act` turns a `respond` decision into an `assistant` message. So drafts can accumulate and be critiqued for as long as you like, and `Act` is called exactly once, at the end, on the draft that survived.

Get that wrong and the loop silently doesn't loop. `Observe -> Think -> Act -> ReflectionThink` cycled with `repeat_from` looks reasonable and never reflects even once: `Act` converts the first draft, `Runtime` sees an `assistant` message and returns.

**How many rounds is yours to choose**, which is why `Loop` takes `count` rather than deciding for you. Two drafts here; a harder piece of writing might want four, and a cheap one none at all:


In [3]:
REFINE_ROUNDS = 2   # draft-then-critique passes before committing to an answer

refine = Runtime(loop=[
    Observe(event="Write a one-sentence tagline for a new brand of running shoes."),
    # no Act inside: a draft is prose, there is nothing to execute
    Loop([Think(allow_fork=False), ReflectionThink()], count=REFINE_ROUNDS),
    Think(allow_fork=False),
    Act(),
])

refined = refine.run(Conversation())
for m in refined.messages:
    print(f"- {m['role']:9} {m['content']!r}")
    if m.get("improvements"):
        print(f"             improve: {m['improvements'][:120]}")

- user      'Write a one-sentence tagline for a new brand of running shoes.'
- decision  'SoleMates: Find your perfect stride with every step, in shoes that match your pace and unleash your greatest escape.'
- critique  "The tagline 'SoleMates: Find your perfect stride with every step, in shoes that match your pace and unleash your greatest escape' does answer the original request by providing a sentence that could work as a tagline for a new brand of running shoes. However, it is a bit long and tries to convey multiple ideas at once, which might make it less effective."
             improve: To improve, focus on making the tagline shorter and more concise while still conveying the unique value proposition of t
- decision  ' \n\nA revised version could be: SoleMates: Find your stride.'
- critique  "The revised tagline 'SoleMates: Find your stride' is concise and directly conveys a key benefit of the running shoes, which is a significant improvement over the original tagline. It is shor

Each draft is a `decision`, each critique a `critique`, and only the last one became an `assistant` message. Nothing in the loop tests `good_enough` — the final `Think` simply reads the critiques as context, the same way it reads anything else in the conversation.

Both patterns are the same three ideas — attempt, critique, attempt again — differing only in what sits inside the cycle and how many times you want it to go round. `count` is the parameter that makes that yours to set rather than the pattern's to assume.


## What you built

✓ Watched a tool call fail permanently with `retries=1` and succeed with `retries=3`, from identical starting arguments — a real repair, not a described one

✓ Built `DebugThink`, which sees the JSON schema and every prior failed attempt rather than the conversation — a narrower task than chapter 1's `resolve_tools()` gave the same model, and validated through `extract()` so a malformed repair never reaches the tool

✓ Kept repair out of the conversation entirely: `Act.execute()` returns a result or a permanent failure and nothing else, so retry noise can't bloat the context or trip the doom-loop detector

✓ Built `ReflectionThink` as critique-only — no `final_answer`, not even a `decision` — so the critic can't quietly grade its own rewrite, and so fixes needing another tool call are still possible

✓ Built `Loop`, with `count` and `until` both checked after a complete pass, and an `assistant` message as the one thing that stops it mid-pass

✓ Ran two rounds, a critique, and a re-entry into the same `Loop` via `repeat_from` — with the retry emerging from a `Think` reading a critique, not from any branch written to handle it

**Next:** `04_planning.ipynb` — deciding the whole sequence up front instead of one step at a time, referencing a result that doesn't exist yet, and running the independent parts at the same time.